# Notebook 02 — Baseline Models
**Project**: IndicSenti — Multilingual Indian Sentiment Analysis  
Baselines: TextBlob · VADER · FastText · Logistic Regression (TF-IDF)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

PALETTE = {
    'mint': '#F1F6F4', 'gold': '#FFC801', 'teal': '#114C5A',
    'sage': '#D9E8E2', 'orange': '#FF9932', 'dark': '#172B36',
}
plt.rcParams.update({
    'figure.facecolor': PALETTE['mint'], 'axes.facecolor': PALETTE['mint'],
    'axes.titlecolor': PALETTE['teal'], 'axes.labelcolor': PALETTE['dark'],
    'text.color': PALETTE['dark'], 'grid.color': PALETTE['sage'],
    'figure.dpi': 120,
})

DATA_DIR = Path('../data')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

X_train, y_train = train_df['text'].astype(str), train_df['label']
X_test,  y_test  = test_df['text'].astype(str),  test_df['label']
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
# ── Baseline 1: TextBlob ────────────────────────────────────
from textblob import TextBlob

def textblob_predict(text):
    polarity = TextBlob(str(text)).sentiment.polarity
    if polarity > 0.1:  return 0  # positive
    if polarity < -0.1: return 2  # negative
    return 1  # neutral

tb_preds = X_test.apply(textblob_predict)
tb_f1 = f1_score(y_test, tb_preds, average='macro')
print(f'TextBlob Macro-F1: {tb_f1:.4f}')
print(classification_report(y_test, tb_preds, target_names=['Positive','Neutral','Negative']))

In [ ]:
# ── Baseline 2: VADER ────────────────────────────────────────
import nltk
nltk.download('vader_lexicon', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sid = SentimentIntensityAnalyzer()

def vader_predict(text):
    scores = sid.polarity_scores(str(text))
    c = scores['compound']
    if c >= 0.05:  return 0
    if c <= -0.05: return 2
    return 1

vader_preds = X_test.apply(vader_predict)
vader_f1 = f1_score(y_test, vader_preds, average='macro')
print(f'VADER Macro-F1: {vader_f1:.4f}')
print(classification_report(y_test, vader_preds, target_names=['Positive','Neutral','Negative']))

In [ ]:
# ── Baseline 3: TF-IDF + Logistic Regression ─────────────────
lr_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1, 2),
                               analyzer='char_wb', min_df=3)),
    ('clf',   LogisticRegression(max_iter=1000, C=1.0,
                                  class_weight='balanced', random_state=42)),
])
lr_pipe.fit(X_train, y_train)
lr_preds = lr_pipe.predict(X_test)
lr_f1 = f1_score(y_test, lr_preds, average='macro')
print(f'TF-IDF + LR Macro-F1: {lr_f1:.4f}')
print(classification_report(y_test, lr_preds, target_names=['Positive','Neutral','Negative']))

In [ ]:
# ── Baseline 4: FastText ─────────────────────────────────────
import fasttext
import tempfile, os

# Write FastText-formatted training file
ft_train_path = '/tmp/ft_train.txt'
with open(ft_train_path, 'w', encoding='utf-8') as f:
    for text, label in zip(X_train, y_train):
        f.write(f'__label__{label} {text}\n')

ft_model = fasttext.train_supervised(
    ft_train_path, epoch=25, lr=0.5, wordNgrams=2,
    dim=100, loss='softmax', thread=4
)

def ft_predict(text):
    label, _ = ft_model.predict(str(text).replace('\n', ' '))
    return int(label[0].replace('__label__', ''))

ft_preds = X_test.apply(ft_predict)
ft_f1 = f1_score(y_test, ft_preds, average='macro')
print(f'FastText Macro-F1: {ft_f1:.4f}')
print(classification_report(y_test, ft_preds, target_names=['Positive','Neutral','Negative']))

In [ ]:
# ── Fig 6: Baseline comparison bar chart ────────────────────
baselines = {
    'TextBlob': tb_f1,
    'VADER':    vader_f1,
    'TF-IDF+LR': lr_f1,
    'FastText': ft_f1,
    'IndicBERT\n+LoRA (ours)': 0.851,  # from paper
}

fig, ax = plt.subplots(figsize=(9, 5))
colors = [PALETTE['sage']] * 4 + [PALETTE['teal']]
bars = ax.bar(baselines.keys(), baselines.values(), color=colors, edgecolor='white', width=0.6)

for bar, val in zip(bars, baselines.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontweight='bold', color=PALETTE['dark'])

ax.set_ylabel('Macro-F1')
ax.set_ylim(0, 1.0)
ax.set_title('Figure 6 — Baseline vs Proposed Model (Macro-F1)')
ax.axhline(0.85, color=PALETTE['orange'], linestyle='--', linewidth=1.5, label='Our Target (0.85)')
ax.legend(); ax.grid(axis='y', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../reports/fig6_baseline_comparison.png', bbox_inches='tight')
plt.show()

print('\nBaseline Summary:')
pd.DataFrame({'Model': list(baselines.keys()), 'Macro-F1': list(baselines.values())}
             ).set_index('Model').round(4)

In [ ]:
# ── Fig 7: Confusion matrix for best baseline ────────────────
best_preds = lr_preds  # TF-IDF+LR is best baseline
cm = confusion_matrix(y_test, best_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns_cmap = sns.diverging_palette(220, 20, as_cmap=True)
im = ax.imshow(cm, cmap='YlGn')

labels = ['Positive', 'Neutral', 'Negative']
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(labels); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Figure 7 — TF-IDF+LR Confusion Matrix')

for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontweight='bold', color='white' if cm[i,j] > cm.max()/2 else PALETTE['dark'])

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('../reports/fig7_baseline_confusion.png', bbox_inches='tight')
plt.show()